In [ ]:
!pwd

In [1]:
import sys
sys.path.append('/home/gpatane/SDO_downstream')
from mae_project.mae.models_mae_2 import mae_model_channel_masking_9ch_with_temporal_attn
import torch
from dataset import SDO_9Channel_Dataset
from torch.utils.data import DataLoader
from utils_2 import load_checkpoint_with_channel_adaptation, freeze_encoder
# Crea il modello con 2 canali di output
model = mae_model_channel_masking_9ch_with_temporal_attn(out_chans=2)
checkpoint_path = '/home/gpatane/SDO_downstream/mae_project/checkpoints/checkpoint_epoch55.pth'

# Carica il checkpoint con adattamento dei pesi
model = load_checkpoint_with_channel_adaptation(
    model, 
    checkpoint_path, 
    in_chans=9, 
    out_chans=2, 
    device='cuda'
)

model = freeze_encoder(model)
model = model.to('cuda')

zarr_path = "/home/gpatane/Dataset/zarr_file_magnetogram_1024_definitivo.zarr"
val_years   = list(range(2021, 2023,3))
wavelengths = ['1700A', '1600A', '335A', '304A', '211A', '193A', '171A', '131A', 'Magnetogram']
val_ds = SDO_9Channel_Dataset(zarr_path, val_years, wavelengths, target_size=1024)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=4)

checkpoint_path = '/home/gpatane/SDO_downstream/seg_project/new_segmentation_model.pth'
model.load_state_dict(torch.load(checkpoint_path))
model.eval()


/home/gpatane/miniconda3/envs/SDOenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Load] Checkpoint ha 9 canali, modello target ne aspetta 2
[Resize] Ridotto decoder_pred: torch.Size([2304, 512]) → torch.Size([512, 512])
[Load] ✓ Checkpoint caricato correttamente
✓ Encoder congelato (frozen)
  Parametri totali: 121,103,104
  Parametri trainabili: 31,129,600 (25.7%)
  Parametri congelati: 89,973,504 (74.3%)


/home/gpatane/miniconda3/envs/SDOenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


MaskedAutoencoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(9, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (fc2): Linear(in_features=3072, out_features=768

In [3]:

import monai
criterion = monai.losses.TverskyLoss(to_onehot_y=True, softmax=True, alpha=0.5, beta=0.5, include_background=False)
from tqdm import tqdm
loader = val_loader
device = 'cuda'
max_grad_norm = 1
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
model.train()
epoch_loss = 0
step = 0
for epochs in range(10):
    for batch in tqdm(loader, desc="Training"):
        data = batch['image'].to(device)
        labels = batch['mask'].to(device)
        optimizer.zero_grad()
        _,pred,_ = model(data)
        recon = model.unpatchify(pred)
        loss = criterion(recon, labels)                                
        loss.backward()
        
        # Gradient clipping per stabilità
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        
        optimizer.step()
        
        epoch_loss += loss.item()
        step += 1

    if scheduler:
        scheduler.step()

    epoch_loss /= step
    print(f"Epoch {epochs+1} - Loss: {epoch_loss:.4f}")


Training: 100%|█████████████████| 181/181 [10:07<00:00,  3.35s/it]


Epoch 1 - Loss: 0.9999


Training: 100%|█████████████████| 181/181 [14:14<00:00,  4.72s/it]


Epoch 2 - Loss: 0.5019


Training: 100%|█████████████████| 181/181 [14:03<00:00,  4.66s/it]


...

Epoch 8 - Loss: 0.1194


Training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 181/181 [14:12<00:00,  4.71s/it]


Epoch 9 - Loss: 0.1063


Training: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 181/181 [13:56<00:00,  4.62s/it]

Epoch 10 - Loss: 0.0956


In [5]:
#save model
torch.save(model.state_dict(), '/home/gpatane/SDO_downstream/seg_project/new_segmentation_model.pth')

In [3]:
from monai.metrics import DiceMetric
from utils_2 import testing
dice_metric = DiceMetric(include_background=False, reduction="mean")
dice_metric_T = DiceMetric(include_background=True, reduction="mean")
device = 'cuda'
metric_dice, metric_dice_T, metric_iou, metric_iou_T = testing(model, val_loader, device, dice_metric, dice_metric_T)

print(f"Dice Score (no background): {metric_dice:.4f}")
print(f"Dice Score (with background): {metric_dice_T:.4f}")
print(f"IoU Score (no background): {metric_iou:.4f}")
print(f"IoU Score (with background): {metric_iou_T:.4f}")

Testing:   0%|                                                                                                                 | 0/181 [00:00<?, ?it/s]/home/gpatane/miniconda3/envs/SDOenv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Testing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 181/181 [11:30<00:00,  3.82s/it]

Dice Score (no background): 0.5987
Dice Score (with background): 0.7993
IoU Score (no background): 0.1718
IoU Score (with background): 0.5858


In [ ]:
import sys

import sunpy
from sunpy.map import Map
import torch.nn as nn
import torch
from torch import optim 

from torch.utils.data import DataLoader

torch.manual_seed(5)
import numpy as np

from monai.transforms import (
    Resized,
    
    Compose,
    EnsureTyped,
    
    RandScaleIntensityd,
    RandShiftIntensityd,
    AsDiscrete,
    ToTensord
)
from monai.metrics import DiceMetric
from monai.losses import DiceLoss
import random

from models import  MAESegmentationModel
from mae.models_mae_2 import mae_for_segmentation_7 

from dataset import SDOMosaicZarrDataset_2, PhotosphereDataset


import matplotlib.pyplot as plt
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
mae_model = mae_for_segmentation_7().to(device)
#checkpoint funzionante pathc_size 14
#mae_model.load_state_dict(torch.load("/home/gpatane/checkpoints/mae_project/new_mae_magnetogram.pth", map_location=device))
#mae_model.load_state_dict(torch.load(model_mae_path, map_location=device)["model_state_dict"])

model_mae_path = '/home/gpatane/checkpoints/mae_project/grid_patch_7_adam.pth'
#mae_model.load_state_dict(torch.load(model_mae_path, map_location=device))

model = MAESegmentationModel(mae_model, num_classes=2, freeze_encoder=True, 
                            decoder_type='deep', dropout=0.1)
#checkpoint_path = 'finetune_segmentation_model_Magnetogram.pth'    #funziona

checkpoint_path = '/home/gpatane/checkpoints/seg_project/NUOVO_RUN_deep_decoder_model_improved.pth'
checkpoint_path = '/home/gpatane/checkpoints/seg_project/anotherNew_checkpoint.pth'
model.load_state_dict(torch.load(checkpoint_path, weights_only=False, map_location=device)["model_state_dict"])
model = model.to(device)

In [ ]:
def plot_grid_to_image(img_batch, colormaps, title='', n_images=3, image_size=224):
    """
    Crea una griglia di immagini con diverse colormap e restituisce
    il risultato come un singolo array NumPy (immagine RGBA).
    """
    
    # 1. Creazione della figura (come prima)
    fig, axes = plt.subplots(n_images, n_images, figsize=(6, 6))
    plt.subplots_adjust(wspace=0, hspace=0, left=0, right=1, bottom=0, top=0.95)

    for i in range(n_images):
        for j in range(n_images):
            sub_img = img_batch[i*image_size:(i+1)*image_size, j*image_size:(j+1)*image_size]
            idx = i * n_images + j
            axes[i, j].imshow(sub_img, cmap=colormaps[idx])
            axes[i, j].axis('off')
            
            for spine in axes[i, j].spines.values():
                spine.set_visible(False)
    
    if title:
        fig.suptitle(title, fontsize=16, y=0.98)
    
    # Adattamento finale dei margini
    fig.subplots_adjust(left=0, right=1, bottom=0, top=0.95 if title else 1)
    
    # --- Modifiche per restituire un'immagine ---
    
    # 2. Forza il rendering della canvas
    fig.canvas.draw()
    
    # 3. Ottieni le dimensioni della canvas (in pixel)
    width, height = fig.canvas.get_width_height()
    
    # 4. Estrai i pixel come buffer RGBA e convertili in array NumPy
    #    Forma risultante: (height, width, 4)
    buffer_rgba = fig.canvas.buffer_rgba()
    image_array = np.frombuffer(buffer_rgba, dtype=np.uint8).reshape((height, width, 4))
    
    # 5. Chiudi la figura per liberare memoria
    plt.close(fig)
    
    # 6. Restituisci l'array dell'immagine
    return image_array
wavelengths = ['1700A', '1600A', '335A', '304A', '211A', '193A', '171A', '131A', 'Magnetogram']
#wavelengths = ['1600A', '304A', '171A', 'Magnetogram']

colormaps=[]
for wl in wavelengths:
    if wl == 'Magnetogram':
        colormaps.append('gray')
    else:
        colormaps.append('sdoaia' + wl.replace('A', ''))
        

def visualize_batch(loader):
    batch = next(iter(loader))
    image = batch["image"]
    mask = batch["mask"]
    no_limb = batch["ic_no_limb_dark"]
    images = image[0].numpy()[0,:,:]
    img = plot_grid_to_image(images, colormaps)
    print(img.shape)
    #images=image.squeeze(0).squeeze(0).cpu().numpy()[0,:,:]
    label = mask.squeeze(0).squeeze(0).cpu().numpy()[0,:,:]
    no_limb = no_limb.squeeze(0).squeeze(0).cpu().numpy()[0,:,:]

    fig = plt.figure(frameon=False)
    ax = plt.Axes(fig, [0.6, 0.6, 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(no_limb[0], cmap="gray")
    
    # plt.figure(figsize=(10, 10))
    # plt.subplot(1, 3, 1)
    # #plt.imshow(images[0], cmap='gray')
    # plt.imshow(img, cmap='gray')
    # plt.axis('off')
    # plt.title('Data')

    # plt.subplot(1, 3, 2)
    # plt.imshow(no_limb[0], cmap='gray')
    # plt.axis('off')
    # plt.title('No Limb Darkening')
    # plt.tight_layout()
    
    # plt.subplot(1, 3, 3)
    # plt.imshow(label[0], cmap='gray')
    # plt.axis('off')
    # plt.title('Label')
    

    plt.show()
    
    return label[0], images[0], no_limb[0]

In [ ]:
start_period = 2010
end_period = 2026
random.seed(5)
all_years = list(range(start_period, end_period))
random.shuffle(all_years)

train_split = int(0.7 * len(all_years))
val_split = int(0.85 * len(all_years))

train_years = list(range(2010,2021))
val_years   = list(range(2021,2023))
test_years  = list(range(2023,2026))


print(f"Train years: {train_years}")
print(f"Validation years: {val_years}")
print(f"Test years: {test_years}")
#tolgo 2018 da train


transform = Compose([
    EnsureTyped(keys=["image", "mask"]),
    ToTensord(keys=["image", "mask"]),
    Resized(keys="image", spatial_size=[672, 672], mode="area"),
    Resized(keys=["mask", "ic_no_limb_dark"], spatial_size=[224, 224], mode="nearest"),
    

])
zarr_path = "/home/gpatane/Dataset/zarr_file_magnetogram.zarr"

wavelengths = ['1700A', '1600A', '335A', '304A', '211A', '193A', '171A', '131A', 'Magnetogram', "Ic_noLimbDark"]
tr_dataset = SDOMosaicZarrDataset_2(zarr_path, train_years, wavelengths, target_size=224, transform=transform)
vl_dataset = SDOMosaicZarrDataset_2(zarr_path, val_years, wavelengths, target_size=224, transform=transform)
ts_dataset = SDOMosaicZarrDataset_2(zarr_path, test_years, wavelengths, target_size=224, transform=transform)
t_loader = DataLoader(tr_dataset, batch_size=12, shuffle=True, num_workers=8)
v_loader = DataLoader(vl_dataset, batch_size=12, shuffle=False, num_workers=4)
ts_loader = DataLoader(ts_dataset, batch_size=12, shuffle=True, num_workers=4)
#_,_,_ = visualize_batch(v_loader)

tot_immagini = len(tr_dataset)+len(vl_dataset)+len(ts_dataset)


print("Train data", len(tr_dataset), "ratio", round(len(tr_dataset)/tot_immagini,3))
print("Validation data", len(vl_dataset), "ratio", round(len(vl_dataset)/tot_immagini,3))
print("Test data", len(ts_dataset), "ratio", round(len(ts_dataset)/tot_immagini,3))

In [ ]:

# --- Carica immagine ---
from PIL import Image, ImageChops, ImageDraw
import os

def remove_borders(img, bg_color=None):
    if bg_color is None:
        bg_color = img.getpixel((0, 0))
    bg = Image.new(img.mode, img.size, bg_color)
    diff = ImageChops.difference(img, bg)
    bbox = diff.getbbox()
    return img.crop(bbox) if bbox else img

# --- Carica immagine ---
img_path = "/home/gpatane/first_project/SDO_downstream/output.png"  # <-- metti qui il percorso della tua immagine
img = Image.open(img_path)

# --- Rimuove bordi automatico ---
img = remove_borders(img)

width, height = img.size
tile_w = width // 3
tile_h = height // 3

os.makedirs("output_tiles", exist_ok=True)
tiles = []

# --- Taglia 3x3 e aggiungi bordo inferiore ---
for row in range(3):
    for col in range(3):
        left = col * tile_w
        upper = row * tile_h
        right = left + tile_w
        lower = upper + tile_h
        tile = img.crop((left, upper, right, lower))

        # Aggiungi bordo inferiore (nero spesso 5px)
        bordered_tile = Image.new("RGB", (tile_w, tile_h + 5), "white")
        bordered_tile.paste(tile, (0, 0))

        tile_path = f"output_tiles/tile_{row}_{col}.png"
        bordered_tile.save(tile_path)
        tiles.append(bordered_tile)

# --- Combina in una colonna ---
column_img = Image.new("RGB", (tile_w, (tile_h + 5) * 9))
y_offset = 0
for t in tiles:
    column_img.paste(t, (0, y_offset))
    y_offset += tile_h + 5

column_img.save("immagine_colonna.png")
print("✅ Operazione completata! File salvati in output_tiles/ e immagine_colonna.png")


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
# Assicurati che wandb sia importato se lo usi, altrimenti rimuovi i riferimenti
# import wandb
model = model.to(device)
def _dice_score_numpy(pred, gt, eps=1e-8):
    """Compute Dice score between two binary numpy arrays."""
    pred = pred.astype(np.bool_)
    gt = gt.astype(np.bool_)
    inter = (pred & gt).sum()
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0  # both empty -> perfect
    return 2.0 * inter / (denom + eps)


def test_and_plot(model, dataloader, device, n_images=6, threshold=0.5, use_wandb=False, save_path=None):
    """
    Esegue il modello e plotta:
      - riga 1: input del modello
      - riga 2: ground truth (overlay rosso) su gt_image
      - riga 3: predizione (overlay blu) su gt_image

    Args:
        model: modello PyTorch
        dataloader: DataLoader (deve fornire un dizionario con 'image', 'mask', e 'gt_image')
        device: torch device
        n_images: numero di esempi da mostrare
        threshold: soglia per binarizzare la predizione (binary)
        use_wandb: se True, logga le immagini su wandb
        save_path: se fornito, salva la figura in questo path
    Returns:
        dice_list: lista dei Dice score per immagine
        mean_dice: Dice score medio
    """
    model.eval()
    batch = next(iter(dataloader))
    inputs = batch['image']      # Immagine di input per il modello
    labels = batch['mask']       # Maschera di ground truth
    # --- MODIFICA 1: Estrai l'immagine di sfondo per gli overlay ---
    gt_images = batch['ic_no_limb_dark'] # Immagine di sfondo per visualizzazione

    batch_size = inputs.shape[0]
    n_show = min(n_images, batch_size)

    dice_list = []

    with torch.no_grad():
        outputs = model(inputs.to(device))

        if outputs.dim() >= 4 and outputs.shape[1] > 1:
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1, keepdim=True).float()
        else:
            probs = torch.sigmoid(outputs)
            preds = (probs > threshold).float()

        fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
        if n_show == 1:
            axes = axes.reshape(1, 3)

        for i in range(n_show):
            # Immagine di input (per la prima riga)
            img_input = inputs[i, 0].cpu().numpy()
            # Maschera di ground truth
            gt_mask = labels[i, 0].cpu().numpy().astype(np.uint8)
            # Predizione
            pred_mask = preds[i].squeeze(0).cpu().numpy().astype(np.uint8)

            # Immagine di sfondo per questo campione
            background_img = gt_images[i, 0].cpu().numpy()

            # Calcola il Dice score
            dice_i = _dice_score_numpy(pred_mask, gt_mask)
            dice_list.append(dice_i)

            # Column 1: original input image (quello che vede il modello)
            axes[i, 0].imshow(img_input, cmap='gray')
            axes[i, 0].set_title(f'Input #{i+1}')
            axes[i, 0].axis('off')

            # Column 2: GT overlay on the background image
            axes[i, 1].imshow(background_img, cmap='gray', alpha=0.8)
            axes[i, 1].imshow(gt_mask, cmap='Reds', alpha=0.5, vmin=0, vmax=1)
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')

            # Column 3: Prediction overlay on the background image
            axes[i, 2].imshow(background_img, cmap='gray', alpha=0.8)
            axes[i, 2].imshow(pred_mask, cmap='Blues', alpha=0.5, vmin=0, vmax=1)
            axes[i, 2].set_title(f'Prediction (Dice={dice_i:.3f})')
            axes[i, 2].axis('off')

    plt.tight_layout()
    mean_dice = float(np.mean(dice_list)) if len(dice_list) > 0 else 0.0

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')

    try:
        if use_wandb and 'wandb' in globals() and wandb.run is not None:
            wandb.log({"test/predictions": wandb.Image(fig)})
            wandb.log({"test/dice_per_image": dice_list, "test/dice_mean": mean_dice})
    except Exception as e:
        print(f"Warning: failed to log to wandb: {e}")

    print("Per-image Dice:")
    for idx, d in enumerate(dice_list, 1):
        print(f"  Image {idx}: {d:.4f}")
    print(f"Mean Dice (shown images): {mean_dice:.4f}")

    plt.show()
    plt.close(fig)


    return dice_list, mean_dice

# Small helper call example (uncomment to run):
dice_list, mean_dice = test_and_plot(model, ts_loader, device, n_images=5, threshold=0.5, use_wandb=False)

In [ ]:
import torch
import numpy as np
from tqdm import tqdm

def test_full_dataset(model, dataloader, device, threshold=0.5):
    """
    Testa il modello sull'intero dataset e calcola metriche aggregate.
    
    Args:
        model: modello PyTorch
        dataloader: DataLoader del test set
        device: torch device
        threshold: soglia per binarizzare la predizione (binary)
    
    Returns:
        results: dizionario con metriche aggregate
    """
    model.eval()
    
    all_dice_scores = []
    total_tp = 0  # True Positives
    total_fp = 0  # False Positives
    total_fn = 0  # False Negatives
    total_tn = 0  # True Negatives
    
    n_samples = 0
    n_empty_gt = 0  # Ground truth completamente vuote
    n_empty_pred = 0  # Predizioni completamente vuote
    
    print(f"Testing on {len(dataloader)} batches...")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Testing"):
            inputs = batch['image'].to(device)
            labels = batch['mask'].to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Convert to predictions
            if outputs.dim() >= 4 and outputs.shape[1] > 1:
                probs = torch.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1, keepdim=True).float()
            else:
                probs = torch.sigmoid(outputs)
                preds = (probs > threshold).float()
            
            # Per ogni immagine nel batch
            batch_size = inputs.shape[0]
            for i in range(batch_size):
                gt_mask = labels[i, 0].cpu().numpy().astype(np.bool_)
                pred_mask = preds[i].squeeze(0).cpu().numpy().astype(np.bool_)
                
                # Calcola Dice score
                inter = (pred_mask & gt_mask).sum()
                denom = pred_mask.sum() + gt_mask.sum()
                
                if denom == 0:
                    dice_i = 1.0  # Entrambi vuoti = perfetto
                    n_empty_gt += 1
                else:
                    dice_i = 2.0 * inter / denom
                
                all_dice_scores.append(dice_i)
                
                # Calcola confusion matrix elements
                tp = (pred_mask & gt_mask).sum()
                fp = (pred_mask & ~gt_mask).sum()
                fn = (~pred_mask & gt_mask).sum()
                tn = (~pred_mask & ~gt_mask).sum()
                
                total_tp += tp
                total_fp += fp
                total_fn += fn
                total_tn += tn
                
                if pred_mask.sum() == 0:
                    n_empty_pred += 1
                
                n_samples += 1
    
    # Calcola metriche aggregate
    mean_dice = np.mean(all_dice_scores)
    std_dice = np.std(all_dice_scores)
    median_dice = np.median(all_dice_scores)
    
    # Precision, Recall, F1
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # IoU (Intersection over Union)
    iou = total_tp / (total_tp + total_fp + total_fn) if (total_tp + total_fp + total_fn) > 0 else 0.0
    
    results = {
        'n_samples': n_samples,
        'mean_dice': mean_dice,
        'std_dice': std_dice,
        'median_dice': median_dice,
        'min_dice': np.min(all_dice_scores),
        'max_dice': np.max(all_dice_scores),
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'iou': iou,
        'n_empty_gt': n_empty_gt,
        'n_empty_pred': n_empty_pred,
        'all_dice_scores': all_dice_scores
    }
    
    return results

# Test sul test set completo
print("="*70)
print("TESTING ON FULL TEST SET")
print("="*70)

results = test_full_dataset(model, ts_loader, device, threshold=0.5)

print("\n" + "="*70)
print("TEST RESULTS")
print("="*70)
print(f"Total samples:           {results['n_samples']}")
print(f"Empty ground truths:     {results['n_empty_gt']} ({results['n_empty_gt']/results['n_samples']*100:.1f}%)")
print(f"Empty predictions:       {results['n_empty_pred']} ({results['n_empty_pred']/results['n_samples']*100:.1f}%)")
print("\n" + "-"*70)
print("DICE SCORE STATISTICS:")
print("-"*70)
print(f"Mean Dice:               {results['mean_dice']:.4f}")
print(f"Std Dice:                {results['std_dice']:.4f}")
print(f"Median Dice:             {results['median_dice']:.4f}")
print(f"Min Dice:                {results['min_dice']:.4f}")
print(f"Max Dice:                {results['max_dice']:.4f}")
print("\n" + "-"*70)
print("PIXEL-WISE METRICS:")
print("-"*70)
print(f"Precision:               {results['precision']:.4f}")
print(f"Recall:                  {results['recall']:.4f}")
print(f"F1 Score:                {results['f1_score']:.4f}")
print(f"IoU:                     {results['iou']:.4f}")
print("="*70)

# Plot distribuzione Dice scores
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(results['all_dice_scores'], bins=50, alpha=0.7, color='royalblue', edgecolor='black')
axes[0].axvline(results['mean_dice'], color='red', linestyle='--', linewidth=2, label=f'Mean: {results["mean_dice"]:.4f}')
axes[0].axvline(results['median_dice'], color='green', linestyle='--', linewidth=2, label=f'Median: {results["median_dice"]:.4f}')
axes[0].set_xlabel('Dice Score', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Dice Scores on Test Set', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Boxplot
axes[1].boxplot(results['all_dice_scores'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Dice Score Distribution (Boxplot)', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')
axes[1].set_ylim([0, 1.05])

# Aggiungi statistiche sul boxplot
textstr = f'Mean: {results["mean_dice"]:.4f}\nStd: {results["std_dice"]:.4f}\nMedian: {results["median_dice"]:.4f}'
axes[1].text(0.98, 0.02, textstr, transform=axes[1].transAxes, fontsize=11,
             verticalalignment='bottom', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n✓ Test completato!")

In [ ]:
# Grafico: anni sull'asse X e una sinusoide con picchi nel 2013 e 2024
import numpy as np
import matplotlib.pyplot as plt

# Ricava la lista di anni dal notebook se esiste, altrimenti usa l'intervallo 2010-2025
try:
    years_present = sorted(set(train_years + val_years + test_years))
    train_yrs = sorted(set(train_years))
    val_yrs = sorted(set(val_years))
    test_yrs = sorted(set(test_years))
except Exception:
    try:
        years_present = sorted(set(all_years))
        # If explicit splits aren't available, split by index like before
        n = len(years_present)
        train_split = int(0.7 * n)
        val_split = int(0.85 * n)
        train_yrs = sorted(years_present[:train_split])
        val_yrs = sorted(years_present[train_split:val_split])
        test_yrs = sorted(years_present[val_split:])
    except Exception:
        years_present = list(range(2010, 2026))
        n = len(years_present)
        train_split = int(0.7 * n)
        val_split = int(0.85 * n)
        train_yrs = years_present[:train_split]
        val_yrs = years_present[train_split:val_split]
        test_yrs = years_present[val_split:]

# Use a full contiguous years array for xticks and shading to cover gaps nicely
min_year = int(min(years_present))
max_year = int(max(years_present))
years = np.arange(min_year, max_year + 1)

# Definizione della sinusoide con periodo 11 anni in modo che abbia picchi in 2013 e 2024
period = 11.0
phase = np.pi / 2  # shift per far partire la sinusoide con picco
# y in [-1,1]
y = np.sin(2 * np.pi * (years - 2013) / period + phase)
# Scala in [0,1]
y_scaled = (y + 1.0) / 2.0

# Plot
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(years, y_scaled, marker='o', linestyle='-', color='tab:blue', label='Sinusoide (0-1)')

# Shaded bands per split: draw one band per year in the split
plotted = {'train': False, 'val': False, 'test': False}
for y0 in years:
    left = y0 - 0.5
    right = y0 + 0.5
    if y0 in train_yrs:
        label = 'Train' if not plotted['train'] else None
        ax.axvspan(left, right, color='tab:green', alpha=0.12, label=label)
        plotted['train'] = True
    elif y0 in val_yrs:
        label = 'Validation' if not plotted['val'] else None
        ax.axvspan(left, right, color='tab:orange', alpha=0.12, label=label)
        plotted['val'] = True
    elif y0 in test_yrs:
        label = 'Test' if not plotted['test'] else None
        ax.axvspan(left, right, color='tab:red', alpha=0.12, label=label)
        plotted['test'] = True

# Evidenzia i picchi richiesti
peak_years = np.array([2013, 2024])
peak_vals = (np.sin(2 * np.pi * (peak_years - 2013) / period + phase) + 1.0) / 2.0
ax.scatter(peak_years, peak_vals, color=['red', 'green'], zorder=5)
for py, pv in zip(peak_years, peak_vals):
    ax.annotate(f'{py}', xy=(py, pv), xytext=(0, 8), textcoords='offset points', ha='center', color='black')

ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Anno')
ax.set_ylabel('Valore (normalizzato)')
ax.set_title('Sinusoide con picchi in 2013 e 2024 — bande Train/Val/Test')
ax.grid(alpha=0.3)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
data.head()

In [ ]:
'''
Contents:
Column 1-2: Gregorian calendar date
- Year
- Month
Column 3: Date in fraction of year for the middle of the corresponding month
Column 4: Monthly mean total sunspot number.
Column 5: Monthly mean standard deviation of the input sunspot numbers from individual stations.
Column 6: Number of observations used to compute the monthly mean total sunspot number.
Column 7: Definitive/provisional marker. A blank indicates that the value is definitive. A '*' symbol indicates that the monthly value is still provisional and is subject to a possible revision (Usually the last 3 to 6 months)
'''
import datetime
import seaborn as sns
import pandas as pd 
file_csv = "/home/gpatane/first_project/SDO_downstream/SN_m_tot_V2.0.csv"

data = pd.read_csv(file_csv, sep = ';', comment='#', header=None,
                   names=['Year', 'Month', 'DateFrac', 'SN_m_tot', 'StdDev', 'N_obs', 'Definitive'])
#make distribution over month the lines
#concert month to string name
data = data[data['Year']>= 2010] 
data.head()
plt.figure(figsize=(12, 6))
sns.lineplot(data=data, x='Year', y='SN_m_tot', color='royalblue', lw=1, ci=None)
#plt.title('Monthly Sunspot Number Trend', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Monthly Mean Total Sunspot Number (SN_m_tot)', fontsize=12)
#plt.grid(alpha=0.2)
plt.show()

In [ ]:

train_years = list(range(2010,2021))
val_years   = list(range(2021,2023))
test_years  = list(range(2023,2026))

print(train_years)
print(val_years)
print(test_years)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random

# --- Parametri e split randomico sugli anni ---
start_period = 2010
end_period = 2026   # 2026 escluso → include fino al 2025
random.seed(5)

all_years = list(range(start_period, end_period))
random.shuffle(all_years)

train_split = int(0.7 * len(all_years))
val_split   = int(0.85 * len(all_years))

# train_years = all_years[:train_split]
# val_years   = all_years[train_split:val_split]
# test_years  = all_years[val_split:]

train_years = list(range(2010,2021))
val_years   = list(range(2021,2023))
test_years  = list(range(2023,2026))

# --- Caricamento dati macchie solari ---
file_csv = "/home/gpatane/first_project/SDO_downstream/SN_m_tot_V2.0.csv"
data = pd.read_csv(
    file_csv,
    sep=';',
    comment='#',
    header=None,
    names=['Year', 'Month', 'DateFrac', 'SN_m_tot', 'StdDev', 'N_obs', 'Definitive']
)

# Filtra solo dati dal 2010 in poi
data = data[data['Year'] >= 2010].copy()

# --- Plot ---
fig, ax = plt.subplots(figsize=(15, 6))

# Traccia linea con dati mensili
sns.lineplot(data=data, x='Year', y='SN_m_tot', color='royalblue', lw=1, ax=ax, label='Sunspot number (monthly)')

# Aggiunge bande colorate per Train/Val/Test
years_present = sorted(data['Year'].unique())
plotted = {'train': False, 'val': False, 'test': False}

for y0 in years_present:
    left = y0 - 0.5
    right = y0 + 0.5
    if y0 in train_years:
        lbl = 'Train 70%' if not plotted['train'] else None
        ax.axvspan(left, right, color='tab:green', alpha=0.12, label=lbl)
        plotted['train'] = True
    elif y0 in val_years:
        lbl = 'Validation 13%' if not plotted['val'] else None
        ax.axvspan(left, right, color='tab:orange', alpha=0.12, label=lbl)
        plotted['val'] = True
    elif y0 in test_years:
        lbl = 'Test 17%' if not plotted['test'] else None
        ax.axvspan(left, right, color='tab:red', alpha=0.12, label=lbl)
        plotted['test'] = True

ax.set_xticks(years_present)
ax.set_xticklabels(years_present, rotation=45)
ax.set_xlabel('Year', fontsize=14)
ax.set_ylabel('Monthly Mean Total Sunspot Number (SN_m_tot)', fontsize=14)
ax.set_ylim(0, 200)  # Limite massimo y a 200

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper left', bbox_to_anchor=(1., 1), borderaxespad=0.)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random

# --- Parametri e split sugli anni ---
start_period = 2010
end_period = 2026   # 2026 escluso → include fino al 2025
random.seed(5)

all_years = list(range(start_period, end_period))
random.shuffle(all_years)

# (I valori per train, val e test sono definiti ma non usati ora)
train_years = list(range(2010,2021))
val_years   = list(range(2021,2023))
test_years  = list(range(2023,2026))

# --- Caricamento dati macchie solari ---
file_csv = "/home/gpatane/first_project/SDO_downstream/SN_m_tot_V2.0.csv"
data = pd.read_csv(
    file_csv,
    sep=';',
    comment='#',
    header=None,
    names=['Year', 'Month', 'DateFrac', 'SN_m_tot', 'StdDev', 'N_obs', 'Definitive']
)
data = data[data['Year'] >= 2010].copy()

# --- Plot ---
# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 6))

# Banda grigia per il periodo 2010-2018
#ax.axvspan(2009.5, 2018.5, facecolor='white', alpha=0.3, label='SDOML dataset period')

# Linee verticali tratteggiate per ogni anno, meno evidenti (colore più chiaro, minore alpha e spessore)
for year in sorted(data['Year'].unique()):
    ax.axvline(x=year, linestyle='--', color='lightgray', alpha=0.3, lw=0.8)

# Traccia linea con dati mensili
sns.lineplot(data=data, x='Year', y='SN_m_tot', color='royalblue', lw=1, ax=ax,
             label='Sunspot number (monthly)')

ax.set_xticks(sorted(data['Year'].unique()))
ax.set_xticklabels(sorted(data['Year'].unique()), rotation=45)
ax.set_xlabel('Year', fontsize=14)
ax.set_ylabel('Monthly Mean Total Sunspot Number (SN_m_tot)', fontsize=14)
ax.set_ylim(0, 200)  # Limite massimo y a 200

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right')#, bbox_to_anchor=(1., 1), borderaxespad=0.)
plt.tight_layout()
plt.show()

In [ ]:
from dataset import clv_correction


import zarr
import skimage.transform
import matplotlib.pyplot as plt
from sunpy.net import Fido, attrs as a
from scipy.ndimage import zoom
import numpy as np
from sunpy.map import Map

path = "/home/gpatane/Dataset/zarr_file_magnetogram.zarr"

zarr_data = zarr.open(path, mode='r')

wavelength = "Ic_noLimbDark"


pixel_dictionary = {}
# --- Visualizza le 9 immagini AIA ---
for year in range(2010, 2026):

    data = zarr_data[str(year)][wavelength]
    for i in range(data.shape[0]):
        image = data[i, :, :]
        mask = clv_correction(image)
        pixel_count = np.sum(mask)
        pixel_dictionary[(year, i)] = pixel_count
        #print(f"Year: {year}, Image {i}, Pixel Count in Mask: {pixel_count}")
        #print(image.shape)
        
    
        #mask = clv_correction(image)
#save dictionary su file npy
pixel_count_list = list(pixel_dictionary.values())
pixel_count_list = [x * 8 for x in pixel_count_list]
pixel_dictionary = {k: v for k, v in zip(pixel_dictionary.keys(), pixel_count_list)}
np.save("pixel_count_dictionary.npy", pixel_dictionary)
# years = []
# pixel_counts = []
# for (year, idx), count in pixel_dictionary.items():
#     years.append(year)
#     count *= 8
#     pixel_counts.append(count)


In [ ]:
import numpy as np
pixel_dictionary = np.load("pixel_count_dictionary.npy", allow_pickle=True).item()


In [ ]:
sorted_items = sorted(pixel_dictionary.items())

labels = [f"{year}-{month:02d}" for (year, month), _ in sorted_items]
values = np.array([value for _, value in sorted_items])

# Calcolo media mobile per smoothing (densità)
window_size = 3  # più grande = più liscia
kernel = np.ones(window_size) / window_size
density_line = np.convolve(values, kernel, mode='same')

# Plot
plt.figure(figsize=(10, 5))
plt.plot(labels, values, marker='o', linestyle='--', label='Valori originali')
plt.plot(labels, density_line, linewidth=2, label='Linea di densità (media mobile)')
plt.xticks(rotation=45)
plt.title("Valori con linea di densità")
plt.xlabel("Anno-Mese")
plt.ylabel("Valore")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
#plot the pixel count over years
years = []
pixel_counts = []
for (year, idx), count in pixel_dictionary.items():
    years.append(year)
    count *= 8
    pixel_counts.append(count)
plt.figure(figsize=(12, 6))
sns.lineplot(x=years, y=pixel_counts, marker='o', color='tab:blue', ci=None)
#plt.scatter(years, pixel_counts, alpha=0.6)
plt.title('Pixel Count in CLV Mask over Years', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Pixel Count in Mask', fontsize=12)

plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 6))

# --- Primo asse: Sunspot Number (asse y sinistro in blu) ---
sns.lineplot(data=data, x='Year', y='SN_m_tot', color='royalblue', lw=1.5, ax=ax1, ci=None)
ax1.set_xlabel('Year', fontsize=14)
ax1.set_ylabel('Sunspot Number', color='royalblue', fontsize=14)
ax1.tick_params(axis='y', labelsize=12, colors='royalblue')
ax1.tick_params(axis='x', labelsize=12)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim(0, 200)

# --- Secondo asse: Pixel Count (asse y destro in rosso) ---
ax2 = ax1.twinx()
years_px = [year for (year, idx) in pixel_dictionary.keys()]
pixel_counts = [count * 8 for count in pixel_dictionary.values()]
sns.lineplot(x=years_px, y=pixel_counts, color='darkred', marker='o', ax=ax2, ci=None)
ax2.set_ylabel('Pixel Count in Mask', color='darkred', fontsize=14)
ax2.tick_params(axis='y', labelsize=12, colors='darkred')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# --- Titolo e leggenda ---
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=12)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
fig, ax1 = plt.subplots(figsize=(13, 6))

# --- Primo asse: Sunspot Number (asse y sinistro in blu) ---
sns.lineplot(data=data, x='Year', y='SN_m_tot', color='royalblue', lw=1.5, ax=ax1, ci=None)
ax1.set_xlabel('Year', fontsize=14)
ax1.set_ylabel('Sunspot Number', color='royalblue', fontsize=14)
ax1.tick_params(axis='y', labelsize=12, colors='royalblue')
ax1.tick_params(axis='x', labelsize=12)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylim(0, 200)

# --- Secondo asse: Pixel Count (asse y destro in rosso) ---
ax2 = ax1.twinx()
years_px = [year for (year, idx) in pixel_dictionary.keys()]
pixel_counts = [count * 8 for count in pixel_dictionary.values()]
sns.lineplot(x=years_px, y=pixel_counts, color='darkred', marker='o', ax=ax2, ci=None)
ax2.set_ylabel('Pixel Count in Mask', color='darkred', fontsize=14)
ax2.tick_params(axis='y', labelsize=12, colors='darkred')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=12)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
pixel_dictionary

In [ ]:
lista = list(pixel_dictionary.values())
above_threshold = [value for value in lista if value > 0]
below_threshold = [value for value in lista if value <= 0]
explode = (0.2, 0)  # esplode la prima fetta
wedges, _ , _ = plt.pie(
    [len(above_threshold), len(below_threshold)],
    autopct='%1.1f%%',
    explode=explode,
    shadow=True,
    colors=['lightblue', 'lightgray']
)

# Legenda esterna
plt.legend(
    wedges,                      # ← collega legenda alle fette
    ['w sunspot', 'w/o sunspot'],
    title='Sunspot Presence',
    loc='upper right',
    bbox_to_anchor=(1.3, 1)
)

#plt.title('Sunspot precent')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load pixel dictionary
pixel_dictionary = np.load('/home/gpatane/first_project/SDO_downstream/seg_project/pixel_count_dictionary.npy', allow_pickle=True).item()

# Define year splits
train_years = list(range(2010, 2021))
val_years = list(range(2021, 2023))
test_years = list(range(2023, 2026))

def get_sunspot_counts(years, pixel_dict):
    """
    Count samples with and without sunspots for given years.
    
    Args:
        years: list of years to consider
        pixel_dict: dictionary with (year, month) keys and pixel counts
    
    Returns:
        tuple: (count_with_sunspot, count_without_sunspot)
    """
    values = []
    for year in years:
        for month in range(12):  # 0-11 months
            key = (year, month)
            if key in pixel_dict:
                values.append(pixel_dict[key])
    
    above_threshold = [v for v in values if v > 0]
    below_threshold = [v for v in values if v <= 0]
    
    return len(above_threshold), len(below_threshold)

# Get counts for each split
train_with, train_without = get_sunspot_counts(train_years, pixel_dictionary)
val_with, val_without = get_sunspot_counts(val_years, pixel_dictionary)
test_with, test_without = get_sunspot_counts(test_years, pixel_dictionary)

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Common settings
colors = ['lightblue', 'lightgray']
explode = (0.2, 0)

# Plot 1: Training set
wedges1, texts1, autotexts1 = axes[0].pie(
    [train_with, train_without],
    autopct='%1.1f%%',
    explode=explode,
    shadow=True,
    colors=colors,
    startangle=90
)
axes[0].set_title(f'Training Set\n({train_years[0]}-{train_years[-1]})', 
                  fontsize=14, fontweight='bold')
axes[0].legend(
    wedges1,
    [f'w/ sunspot (n={train_with})', f'w/o sunspot (n={train_without})'],
    title='Sunspot Presence',
    loc='upper right',
    bbox_to_anchor=(1.3, 1)
)

# Plot 2: Validation set
wedges2, texts2, autotexts2 = axes[1].pie(
    [val_with, val_without],
    autopct='%1.1f%%',
    explode=explode,
    shadow=True,
    colors=colors,
    startangle=90
)
axes[1].set_title(f'Validation Set\n({val_years[0]}-{val_years[-1]})', 
                  fontsize=14, fontweight='bold')
axes[1].legend(
    wedges2,
    [f'w/ sunspot (n={val_with})', f'w/o sunspot (n={val_without})'],
    title='Sunspot Presence',
    loc='upper right',
    bbox_to_anchor=(1.3, 1)
)

# Plot 3: Test set
wedges3, texts3, autotexts3 = axes[2].pie(
    [test_with, test_without],
    autopct='%1.1f%%',
    explode=explode,
    shadow=True,
    colors=colors,
    startangle=90
)
axes[2].set_title(f'Test Set\n({test_years[0]}-{test_years[-1]})', 
                  fontsize=14, fontweight='bold')
axes[2].legend(
    wedges3,
    [f'w/ sunspot (n={test_with})', f'w/o sunspot (n={test_without})'],
    title='Sunspot Presence',
    loc='upper right',
    bbox_to_anchor=(1.3, 1)
)

# Make percentage text more readable
for autotext in autotexts1 + autotexts2 + autotexts3:
    autotext.set_color('black')
    autotext.set_fontsize(11)
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.savefig('/home/gpatane/first_project/SDO_downstream/seg_project/data_distribution_piecharts.png', 
            dpi=300, bbox_inches='tight')
print("✓ Pie charts saved to: seg_project/data_distribution_piecharts.png")

# Print summary statistics
print("\n" + "="*60)
print("DATA DISTRIBUTION SUMMARY")
print("="*60)
print(f"\nTraining Set ({train_years[0]}-{train_years[-1]}):")
print(f"  With sunspot:    {train_with:>4} samples ({train_with/(train_with+train_without)*100:.1f}%)")
print(f"  Without sunspot: {train_without:>4} samples ({train_without/(train_with+train_without)*100:.1f}%)")
print(f"  Total:           {train_with+train_without:>4} samples")

print(f"\nValidation Set ({val_years[0]}-{val_years[-1]}):")
print(f"  With sunspot:    {val_with:>4} samples ({val_with/(val_with+val_without)*100:.1f}%)")
print(f"  Without sunspot: {val_without:>4} samples ({val_without/(val_with+val_without)*100:.1f}%)")
print(f"  Total:           {val_with+val_without:>4} samples")

print(f"\nTest Set ({test_years[0]}-{test_years[-1]}):")
print(f"  With sunspot:    {test_with:>4} samples ({test_with/(test_with+test_without)*100:.1f}%)")
print(f"  Without sunspot: {test_without:>4} samples ({test_without/(test_with+test_without)*100:.1f}%)")
print(f"  Total:           {test_with+test_without:>4} samples")

total_with = train_with + val_with + test_with
total_without = train_without + val_without + test_without
total_samples = total_with + total_without

print(f"\nOverall:")
print(f"  With sunspot:    {total_with:>4} samples ({total_with/total_samples*100:.1f}%)")
print(f"  Without sunspot: {total_without:>4} samples ({total_without/total_samples*100:.1f}%)")
print(f"  Total:           {total_samples:>4} samples")
print("="*60)

plt.show()


# Distribuzione Sunspot per Train/Val/Test Sets

Analizziamo la distribuzione delle entità con e senza sunspot per ogni set di dati.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Carica il dizionario
pixel_dictionary = np.load('pixel_count_dictionary.npy', allow_pickle=True).item()

# Definisci gli anni per ogni set
train_years = list(range(2010, 2021))
val_years = list(range(2021, 2023))
test_years = list(range(2023, 2026))

# Funzione per contare entità con/senza sunspot per un set di anni
def count_sunspot_presence(pixel_dict, years):
    with_sunspot = 0
    without_sunspot = 0
    
    for key, pixel_count in pixel_dict.items():
        year, day = key
        if year in years:
            if pixel_count > 0:
                with_sunspot += 1
            else:
                without_sunspot += 1
    
    return with_sunspot, without_sunspot

# Conta per ogni set
train_with, train_without = count_sunspot_presence(pixel_dictionary, train_years)
val_with, val_without = count_sunspot_presence(pixel_dictionary, val_years)
test_with, test_without = count_sunspot_presence(pixel_dictionary, test_years)

print(f"Train Set: {train_with} con sunspot, {train_without} senza sunspot (Total: {train_with + train_without})")
print(f"Val Set: {val_with} con sunspot, {val_without} senza sunspot (Total: {val_with + val_without})")
print(f"Test Set: {test_with} con sunspot, {test_without} senza sunspot (Total: {test_with + test_without})")

# Crea 3 pie charts
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

datasets = [
    ('Train Set', [train_with, train_without], train_with + train_without),
    ('Validation Set', [val_with, val_without], val_with + val_without),
    ('Test Set', [test_with, test_without], test_with + test_without)
]

explode = (0.1, 0)  # esplode la prima fetta
colors = ['lightblue', 'lightgray']

for idx, (title, counts, total) in enumerate(datasets):
    wedges, texts, autotexts = axes[idx].pie(
        counts,
        autopct='%1.1f%%',
        explode=explode,
        shadow=True,
        colors=colors,
        startangle=90
    )
    
    # Migliora la leggibilità dei percentuali
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontsize(12)
        autotext.set_weight('bold')
    
    axes[idx].set_title(f'{title}\n(Total: {total} samples)', fontsize=14, fontweight='bold')
    
    # Legenda solo per il primo grafico
    if idx == 0:
        axes[idx].legend(
            wedges,
            [f'With Sunspot ({counts[0]})', f'Without Sunspot ({counts[1]})'],
            title='Sunspot Presence',
            loc='upper left',
            bbox_to_anchor=(-0.3, 1)
        )

plt.tight_layout()
plt.show()

# Stampa statistiche dettagliate
print("\n" + "="*60)
print("SUNSPOT DISTRIBUTION STATISTICS")
print("="*60)
for title, counts, total in datasets:
    with_sp, without_sp = counts
    percentage_with = (with_sp / total * 100) if total > 0 else 0
    print(f"\n{title}:")
    print(f"  - With Sunspot: {with_sp} ({percentage_with:.1f}%)")
    print(f"  - Without Sunspot: {without_sp} ({100-percentage_with:.1f}%)")
    print(f"  - Total: {total}")
print("="*60)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# Assumiamo che la tua UNet accetti gli argomenti: in_channels, out_channels, bilinear
from unet_pytorch.model import UNet 
import random

class MaskedAutoencoderUNet(nn.Module):
    def __init__(self, img_size=1024, in_chans=9):
        super().__init__()
        self.img_size = img_size
        self.in_chans = in_chans
        
        # CORREZIONE LOGICA 1:
        # L'input della U-Net è: Immagine (in_chans) + Maschera (in_chans) = in_chans * 2
        unet_input_channels = in_chans * 2
        
        # CORREZIONE LOGICA 2:
        # L'output della U-Net deve essere l'immagine ricostruita completa (in_chans)
        unet_output_channels = in_chans
        
        self.unet = UNet(in_channels=unet_input_channels, out_channels=unet_output_channels)
        
    def generate_channel_mask(self, x, num_to_mask):
        """
        Genera una maschera binaria [N, C, H, W].
        1 = Canale Mascherato (da ricostruire)
        0 = Canale Visibile (input)
        """
        N, C, H, W = x.shape
        mask = torch.zeros((N, C, H, W), device=x.device)
        
        # Logica randomizzazione
        if num_to_mask is None:
            # Se non specificato, maschera un numero random di canali (da 1 a C-1)
            # Non ha senso mascherare 0 canali o TUTTI i canali (input tutto nero)
            k = random.randint(1, C - 1)
        else:
            k = num_to_mask
            
        for i in range(N):
            # Sceglie k indici di canali a caso da mascherare
            # randperm restituisce indici mescolati, prendiamo i primi k
            masked_indices = torch.randperm(C)[:k]
            mask[i, masked_indices, :, :] = 1.0
            
        return mask

    def forward(self, imgs, num_mask_channels=None):
        """
        imgs: [N, C, H, W]
        num_mask_channels: (int) numero di canali da nascondere. Se None, è random.
        """
        # 1. Genera Maschera [N, C, H, W]
        mask = self.generate_channel_mask(imgs, num_to_mask=num_mask_channels)
        
        # 2. Maschera l'immagine
        # Dove mask è 1, l'immagine diventa 0.
        imgs_masked = imgs * (1 - mask)
        
        # 3. Concatenazione
        # L'input alla rete è: [Canali Visibili (con zeri nei buchi), Maschera Binaria]
        # Dimensione totale canali: 9 + 9 = 18
        x_input = torch.cat([imgs_masked, mask], dim=1)
        
        # 4. Forward pass attraverso la U-Net
        # Ora la U-Net accetta correttamente 18 canali e ne sputa fuori 9
        pred = self.unet(x_input)
        
        # 5. Calcolo della Loss
        loss = self.forward_loss(imgs, pred, mask)
        
        return loss, pred, mask

    def forward_loss(self, imgs, pred, mask):
        """
        Calcola MSE loss solo sui canali mascherati.
        """
        # Calcola differenza al quadrato per tutti i pixel
        loss = (pred - imgs) ** 2
        
        # Moltiplica per la maschera.
        # Questo azzera la loss sui canali che erano visibili (dove mask=0).
        # Vogliamo che la rete impari SOLO a ricostruire ciò che non ha visto.
        loss = loss * mask
        
        # Somma di tutti gli errori / numero di pixel mascherati
        # Aggiungiamo epsilon per stabilità numerica (evitare divisione per 0)
        loss = loss.sum() / (mask.sum() + 1e-6)
        
        return loss






# Esempio di utilizzo corretto
def test_unet_mae(batch):
    # Nota: Rimosso 'out_channels' dall'init perché non serve più
    model = MaskedAutoencoderUNet(img_size=256, in_chans=9) 
    
    img = batch['image']
    
    # Forward pass (sceglierà un numero random di canali da mascherare)
    loss, pred, mask = model(img)
    
    print(f"Input shape: {img.shape}")      # [2, 9, 256, 256]
    print(f"Prediction shape: {pred.shape}") # [2, 9, 256, 256]
    print(f"Mask shape: {mask.shape}")       # [2, 9, 256, 256]
    print(f"MAE Loss: {loss.item()}")

def train(loader, model, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        imgs = batch['image'].to(device)
        
        optimizer.zero_grad()
        loss, pred, mask = model(imgs)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader)
    return avg_loss
from dataset import SDO_9Channel_Dataset
from torch.utils.data import DataLoader
zarr_path = "/home/gpatane/Dataset/zarr_file_magnetogram.zarr"
wavelengths = ['1700A', '1600A', '335A', '304A', '211A', '193A', '171A', '131A', 'Magnetogram']
image_size = 256
batch_size = 1 
train_years = list(range(2011, 2021))
train_ds = SDO_9Channel_Dataset(zarr_path, train_years, wavelengths, target_size=image_size)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

batch = next(iter(train_loader))

test_unet_mae(batch)

In [ ]:
import numpy as np
import torch
import zarr
import pandas as pd
from torch.utils.data import DataLoader, Subset

def get_day_loader(zarr_path, target_year, target_date_str, wavelengths, batch_size=4, num_workers=2):
    """
    Crea un DataLoader che carica solo i campioni di una specifica giornata.
    
    Args:
        zarr_path (str): Path al file Zarr.
        target_year (str): L'anno in cui cercare (es. '2014').
        target_date_str (str): La data da filtrare (es. '2014-05-12').
        wavelengths (list): Lista canali.
    """
    
    # 1. Apriamo il Zarr per ispezionare le date SENZA caricare le immagini
    #    Nota: questo passaggio è molto veloce perché leggiamo solo metadati/testo.
    root = zarr.open(zarr_path, mode='r')
    
    # Assumiamo che dentro l'anno ci sia un array 'T_OBS' o simile con i timestamp
    # Modifica 'T_OBS' con il nome corretto nel tuo Zarr (es. 'time', 'DATE-OBS')
    if 'T_OBS' not in root[target_year]:
        raise ValueError(f"Chiave 'T_OBS' non trovata in {zarr_path}/{target_year}. Verifica il nome dell'array temporale.")
        
    timestamps = root[target_year]['T_OBS'][:] # Carica tutte le date in memoria (è leggero)
    
    # 2. Troviamo gli indici (local_idx) che corrispondono al giorno target
    #    Utilizziamo pandas per gestire facilmente formati data diversi
    ts_series = pd.to_datetime(timestamps)
    
    # Creiamo una maschera booleana per il giorno specifico
    target_dt = pd.to_datetime(target_date_str)
    day_mask = (ts_series.year == target_dt.year) & \
               (ts_series.month == target_dt.month) & \
               (ts_series.day == target_dt.day)
    
    # Otteniamo gli indici numerici (es. [105, 106, 107, ...])
    day_indices = np.where(day_mask)[0]
    
    if len(day_indices) == 0:
        print(f"Attenzione: Nessun dato trovato per il giorno {target_date_str}")
        return None

    print(f"Trovati {len(day_indices)} campioni per il giorno {target_date_str}.")

    # 3. Istanziamo il Dataset COMPLETO (ma solo per l'anno di interesse per efficienza)
    #    Nota: passiamo solo [target_year] così gli indici del dataset (0, 1, 2...) 
    #    corrispondono esattamente agli indici del file Zarr (local_idx).
    full_dataset = SDO_9Channel_Dataset(
        zarr_path=zarr_path, 
        list_year=[target_year],  # Importante: isoliamo l'anno
        wavelengths=wavelengths
    )
    
    # Verifichiamo che gli indici trovati non superino la lunghezza effettiva del dataset
    # (Il dataset calcola il 'min' tra i canali, il vettore T_OBS potrebbe essere più lungo se ci sono file corrotti)
    valid_indices = [idx for idx in day_indices if idx < len(full_dataset)]
    
    # 4. Creiamo un Subset
    #    Il Subset agisce come un wrapper: quando il loader chiede l'indice 0 del subset,
    #    lui va a prendere l'indice 'valid_indices[0]' del full_dataset.
    day_subset = Subset(full_dataset, valid_indices)
    
    # 5. Creiamo il DataLoader
    loader = DataLoader(
        day_subset, 
        batch_size=batch_size, 
        shuffle=False,  # Di solito per sequenze temporali non si shakera, ma dipende dal training
        num_workers=num_workers
    )
    
    return loader

# --- ESEMPIO DI UTILIZZO ---
zarr_path = "/home/gpatane/Dataset/zarr_file_magnetogram_1024_definitivo.zarr"
wavelengths = ['94', '131', '171', '193', '211', '304', '335', '1600', '1700'] # Esempio

loader_maggio_12 = get_day_loader(
    zarr_path=zarr_path, 
    target_year='2024', 
    target_date_str='2024-05-20', 
    wavelengths=wavelengths,
    batch_size=8
)

# Ora puoi iterare
if loader_maggio_12:
    for batch in loader_maggio_12:
        imgs = batch['image'] # [B, 9, 512, 512]
        masks = batch['mask']
        print(f"Batch caricato: {imgs.shape}")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Subset
from dataset import SDO_9Channel_Dataset

def create_loader_by_indices(zarr_path, year, start_idx, end_idx, wavelengths, batch_size=2):
    """
    Crea un loader basato puramente sugli indici numerici, 
    utile quando mancano i metadati temporali nel Zarr.
    """
    
    # 1. Istanziamo il Dataset completo per quell'anno
    dataset = SDO_9Channel_Dataset(
        zarr_path=zarr_path,
        list_year=[year], 
        wavelengths=wavelengths,
        target_size=1024, # o 1024
        num_classes=1
    )
    
    # 2. Creiamo la lista degli indici che vogliamo caricare
    # Assicuriamoci di non uscire dai limiti del dataset
    max_len = len(dataset)
    requested_indices = list(range(start_idx, end_idx + 1))
    valid_indices = [i for i in requested_indices if i < max_len]
    
    if len(valid_indices) == 0:
        print("Errore: Gli indici richiesti sono fuori dal range del dataset.")
        return None
        
    print(f"--- Creazione Loader ---")
    print(f"Anno: {year}")
    print(f"Indici richiesti: {start_idx} -> {end_idx}")
    print(f"Campioni effettivi: {len(valid_indices)}")
    
    # 3. Creiamo il Subset
    subset = Subset(dataset, valid_indices)
    
    # 4. Loader
    loader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=False, 
        num_workers=2
    )
    
    return loader

# --- UTILIZZO PER IL "30 MAGGIO" (STIMATO) ---

zarr_file = "/home/gpatane/Dataset/zarr_file_magnetogram_1024_definitivo.zarr"
wls = ['1700A', '1600A', '335A', '304A', '211A', '193A', '171A', '131A', 'Magnetogram']

# Sappiamo che l'indice 300 è circa fine Maggio.
# Prendiamo una "finestra" di 4 immagini intorno a quell'indice per essere sicuri
# di beccare la giornata (es. indici 298, 299, 300, 301)
idx_center = 280
window = 10

loader_maggio = create_loader_by_indices(
    zarr_path=zarr_file,
    year='2024',
    start_idx=idx_center - window, # 298
    end_idx=idx_center + window,   # 302
    wavelengths=wls,
    batch_size=21
)

# Verifica visiva
batch = next(iter(loader_maggio))
batch['image'].shape  # Dovrebbe essere [1, 9, 1024, 1024]  

In [ ]:
from models import MAE_UNet_Segmentation, MAE_Seg_Advanced, MAE_Seg_Deformer
from mae.models_mae_2 import mae_model_channel_masking_9ch_with_temporal_attn
from utils_2 import run_and_plot_predictions_all_channels
from monai.metrics import DiceMetric
device = "cuda"
model_checkpoints = "/home/gpatane/checkpoints/seg_project/checkpoints/1024_2026_01_21_MAE_Seg_Deformer.pth"
mae_backbone = mae_model_channel_masking_9ch_with_temporal_attn().to(device)
model = MAE_Seg_Deformer(mae_backbone, num_classes=2).to(device)
checkpoint = torch.load(model_checkpoints, map_location=device, weights_only=False)
state_dict = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)    
dice_metric = DiceMetric(include_background=False, reduction="mean")
dice_metric_T = DiceMetric(include_background=True, reduction="mean")
save_plot = "/home/gpatane/first_project/SDO_downstream/seg_project/predictions/"


run_and_plot_predictions_all_channels(model, loader_maggio, device, dice_metric=dice_metric, dice_metric_T=dice_metric_T, n_images=20, threshold=0.5, use_wandb=False, save_path=save_plot)